### Example of checking single RTdose file

In [1]:
import SimpleITK as sitk

file_path = "/database/brainmets/dicom/SBRT_research_mim_export_20251209_organized/SRS1655/2002-02__Studies/SRS1655_SRS1655_RTDOSE_2002-02-13_000000_._._n1__00000/2.16.840.1.114362.1.12046989.25631758973.631603306.627.3852.dcm"
rtdose_image = sitk.ReadImage(file_path)

print(rtdose_image)

Image (0x250d6360)
  RTTI typeinfo:   itk::Image<unsigned short, 3u>
  Reference Count: 1
  Modified Time: 1872
  Debug: Off
  Object Name: 
  Observers: 
    none
  Source: (none)
  Source output name: (none)
  Release Data: Off
  Data Released: False
  Global Release Data: Off
  PipelineMTime: 1852
  UpdateMTime: 1868
  RealTimeStamp: 0 seconds 
  LargestPossibleRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [297, 380, 299]
  BufferedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [297, 380, 299]
  RequestedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [297, 380, 299]
  Spacing: [0.5, 0.5, 0.5]
  Origin: [76.3083, 43.7724, 14.3143]
  Direction: 
1 -4.00275e-17 -8.16466e-19
4.00275e-17 1 7.58402e-19
-8.16466e-19 7.58402e-19 -1

  IndexToPointMatrix: 
0.5 -2.00138e-17 -4.08233e-19
2.00138e-17 0.5 3.79201e-19
-4.08233e-19 3.79201e-19 -0.5

  PointToIndexMatrix: 
2 8.00551e-17 -1.63293e-18
-8.00551e-17 2 1.5168e-18
-1.63293e-18 1.5168e-18 -2

  Inverse Direc

### Overview all RTdose file

In [2]:
import os
import csv
from pathlib import Path
import SimpleITK as sitk
from collections import defaultdict
from tqdm import tqdm

# Base directory
base_dir = "/database/brainmets/dicom/SBRT_research_mim_export_20251209_organized"

# Store all dose folder information
dose_folders_info = []

# Walk through the directory structure
print("Scanning for RTDOSE folders...\n")
print("=" * 100)

cnt_patient = 0
cnt_study = 0
cnt_dose = 0

# Get all patient folders first for progress bar
patient_folders = [f for f in sorted(os.listdir(base_dir)) 
                   if os.path.isdir(os.path.join(base_dir, f))]

pbar = tqdm(patient_folders, desc="Processing patients")
for patient_folder in pbar:
    
    cnt_patient += 1
    
    # Update progress bar with current patient name
    pbar.set_postfix(patient=patient_folder)

    patient_path = os.path.join(base_dir, patient_folder)
    
    # Look for study folders (folders containing "__Studies")
    for study_folder in sorted(os.listdir(patient_path)):

        cnt_study += 1

        study_path = os.path.join(patient_path, study_folder)
        
        if not os.path.isdir(study_path):
            continue
        
        # Check if this is a study folder (contains "__Studies")
        if "__Studies" not in study_folder:
            continue
        
        cnt_dose_folder_in_study = 0

        # Look for RTDOSE folders
        for item in sorted(os.listdir(study_path)):
            item_path = os.path.join(study_path, item)
            
            if not os.path.isdir(item_path):
                continue
            
            # Check if this is an RTDOSE folder
            if "RTDOSE" in item:
                
                cnt_dose_folder_in_study += 1

                # Count DICOM files in this dose folder (files that are not directories)
                dicom_files = [f for f in os.listdir(item_path) 
                             if os.path.isfile(os.path.join(item_path, f))]
                num_dose_files = len(dicom_files)

                assert num_dose_files == 1, f"One RTdose folder should only have one RTdose file, but {num_dose_files} files found for Patient {patient_folder} Study {study_folder}"

                
                
                # Print information
                # print(f"Patient: {patient_folder}")
                # print(f"  Study: {study_folder}")
                # print(f"    Dose Folder: {item}")
                # print(f"      Number of RTDOSE files: {num_dose_files}")
                # print()

                
                # Get size and spacing from first DICOM file if available
                size = None
                spacing = None
                
                if dicom_files:
                    try:
                        first_dcm_file = os.path.join(item_path, dicom_files[0])
                        rtdose_image = sitk.ReadImage(first_dcm_file)
                        size = list(rtdose_image.GetSize())
                        spacing = list(rtdose_image.GetSpacing())
                    except Exception as e:
                        print(f"      Warning: Could not read DICOM file: {e}")
                
                # Store information
                dose_folders_info.append({
                    'patient': patient_folder,
                    'study': study_folder,
                    'dose_folder': item,
                    'size': size,
                    'spacing': spacing,
                    'num_files': num_dose_files
                })
            
        cnt_dose += cnt_dose_folder_in_study

        if cnt_dose_folder_in_study != 1:
            if cnt_dose_folder_in_study == 0:
                print(f"No RTDOSE folders found for Patient {patient_folder} Study {study_folder}")
            else:
                print(f"Multiple {cnt_dose_folder_in_study} RTDOSE folders found for Patient {patient_folder} Study {study_folder}")

print("=" * 100)
print(f"Total patients: {cnt_patient}")
print(f"Total studies: {cnt_study}")
print(f"Total doses: {cnt_dose}")
print(f"\nTotal RTDOSE folders found: {len(dose_folders_info)}\n")

# Sort by patient name first, then by study folder
dose_folders_info.sort(key=lambda x: (x['patient'], x['study']))

# Create CSV file
csv_filename = "rtdose_overview.csv"
print(f"Creating CSV file: {csv_filename}")

with open(csv_filename, 'w', newline='') as csvfile:
    fieldnames = ['patient', 'study', 'dose_folder', 'size', 'spacing']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    
    writer.writeheader()
    for info in dose_folders_info:
        writer.writerow({
            'patient': info['patient'],
            'study': info['study'],
            'dose_folder': info['dose_folder'],
            'size': str(info['size']) if info['size'] else '',
            'spacing': str(info['spacing']) if info['spacing'] else ''
        })

print(f"CSV file created successfully with {len(dose_folders_info)} entries.")
print(f"File saved as: {os.path.abspath(csv_filename)}")


Scanning for RTDOSE folders...



Processing patients:   9%|███████████▌                                                                                                                         | 2/23 [00:01<00:10,  1.99it/s, patient=SRS0979]

Multiple 2 RTDOSE folders found for Patient SRS0979 Study 1999-08__Studies


Processing patients:  26%|██████████████████████████████████▋                                                                                                  | 6/23 [00:03<00:11,  1.43it/s, patient=SRS1204]

Multiple 3 RTDOSE folders found for Patient SRS1190 Study 2001-06__Studies
Multiple 5 RTDOSE folders found for Patient SRS1204 Study 1999-08__Studies
Multiple 2 RTDOSE folders found for Patient SRS1204 Study 2000-01__Studies


...

No RTDOSE folders found for Patient SRS1885 Study 2015-10__Studies


Processing patients:  91%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌           | 21/23 [00:16<00:01,  1.32it/s, patient=SRS2443]

Multiple 2 RTDOSE folders found for Patient SRS1885 Study 2015-11__Studies
Multiple 2 RTDOSE folders found for Patient SRS1885 Study 2017-05__Studies


Processing patients: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 23/23 [00:17<00:00,  1.31it/s, patient=SRS2839]

Total patients: 23
Total studies: 72
Total doses: 98

Total RTDOSE folders found: 98

Creating CSV file: rtdose_overview.csv
CSV file created successfully with 98 entries.
File saved as: /homebase/DL_projects/code_sync/myMedIADIRLab/data_io/dicom_reader/rtstruct_debug/rtdose_overview.csv
